# Exploratory Data Analysis: Colombian Electricity Spot Market

Variables analyzed:
1. **Spot price** (hourly, 2000-2026)
2. **Hydro inflows** and **reservoir levels** (daily)
3. **Generation** and **demand** (hourly)

Data source: XM S.A. E.S.P., Sinergox API.

<d.benavidess@uniandes.edu.co>

In [49]:
import numpy as np
import pandas as pd
import altair as alt

alt.data_transformers.enable('vegafusion')

DataTransformerRegistry.enable('vegafusion')

## 1. Load processed data

In [50]:
precio = pd.read_csv("../data/processed/precio_bolsa.csv", parse_dates=["datetime"])
aportes = pd.read_csv("../data/processed/aportes_energia.csv", parse_dates=["date"])
volumen = pd.read_csv("../data/processed/volumen_util_pct.csv", parse_dates=["date"])
generacion = pd.read_csv("../data/processed/generacion_real.csv", parse_dates=["datetime"])
demanda = pd.read_csv("../data/processed/demanda_real.csv", parse_dates=["datetime"])

print(f"Spot price:   {len(precio):>10,} rows  |  {precio.datetime.min().date()} to {precio.datetime.max().date()}")
print(f"Inflows:      {len(aportes):>10,} rows  |  {aportes.date.min().date()} to {aportes.date.max().date()}")
print(f"Reservoir %:  {len(volumen):>10,} rows  |  {volumen.date.min().date()} to {volumen.date.max().date()}")
print(f"Generation:   {len(generacion):>10,} rows  |  {generacion.datetime.min().date()} to {generacion.datetime.max().date()}")
print(f"Demand:       {len(demanda):>10,} rows  |  {demanda.datetime.min().date()} to {demanda.datetime.max().date()}")

Spot price:      233,016 rows  |  2000-01-01 to 2026-07-31
Inflows:           9,709 rows  |  2000-01-01 to 2026-07-31
Reservoir %:       9,709 rows  |  2000-01-01 to 2026-07-31
Generation:      233,016 rows  |  2000-01-01 to 2026-07-31
Demand:          233,016 rows  |  2000-01-01 to 2026-07-31


## 2. Spot price time series

- El Niño 2002-03, 2009-10, 2015-16, 2023-24
- COVID-19 demand drop (March-May 2020)
- Regulatory interventions and scarcity price activations

In [51]:
precio_daily = precio.set_index("datetime").resample("D")["precio_cop_kwh"].mean().reset_index()
precio_daily.columns = ["date", "precio_daily_avg"]

daily_average_spot_price = alt.Chart(precio_daily).mark_line(strokeWidth=0.5).encode(
    x=alt.X("date:T", title=""),
    y=alt.Y("precio_daily_avg:Q", title="Spot price (COP/kWh)"),
).properties(width=800, height=300, title="Daily average spot price (2000-2026)")

daily_average_spot_price

alt.Chart(...)

In [ ]:
daily_average_spot_price.save("../figures/eda/daily_average_spot_price.png", scale_factor=2.0)

In [53]:
daily_avg_spot_price_log = alt.Chart(precio_daily).mark_line(strokeWidth=0.5).encode(
    x=alt.X("date:T", title=""),
    y=alt.Y("precio_daily_avg:Q", title="Spot price (COP/kWh)", scale=alt.Scale(type="log")),
).properties(width=800, height=300, title="Daily average spot price (log scale)")

daily_avg_spot_price_log

alt.Chart(...)

In [54]:
daily_avg_spot_price_log.save("../figures/eda/daily_avg_spot_price_log.png", scale_factor=2.0)

## 3. Price distribution

In [55]:
alt.Chart(precio_daily).mark_bar().encode(
    x=alt.X("precio_daily_avg:Q", bin=alt.Bin(maxbins=100), title="Daily avg price (COP/kWh)"),
    y=alt.Y("count()", title="Days"),
).properties(width=600, height=300, title="Distribution of daily average spot price")

alt.Chart(...)

In [56]:
precio_daily["log_price"] = np.log10(precio_daily["precio_daily_avg"])

alt.Chart(precio_daily).mark_bar().encode(
    x=alt.X("log_price:Q", bin=alt.Bin(maxbins=80), title="log10(price)"),
    y=alt.Y("count()", title="Days"),
).properties(width=600, height=300, title="Distribution of log10(daily avg spot price)")

alt.Chart(...)

In [57]:
precio_daily["precio_daily_avg"].describe()

count    9709.000000
mean      184.426083
std       213.053825
min        28.841420
25%        70.288423
50%       113.004473
75%       196.037733
max      2498.804070
Name: precio_daily_avg, dtype: float64

## 4. Seasonality

### 4.1 Hourly profile

In [58]:
precio["hour"] = precio["datetime"].dt.hour

hourly_profile = precio.groupby("hour")["precio_cop_kwh"].agg(["mean", "median"]).reset_index()
hourly_profile.columns = ["hour", "mean", "median"]
hourly_long = hourly_profile.melt("hour", var_name="statistic", value_name="price")

alt.Chart(hourly_long).mark_line(point=True).encode(
    x=alt.X("hour:O", title="Hour of day"),
    y=alt.Y("price:Q", title="COP/kWh"),
    color="statistic:N",
).properties(width=600, height=300, title="Hourly price profile (2000-2026)")

alt.Chart(...)

### 4.2 Monthly averages

In [59]:
precio_daily["month"] = precio_daily["date"].dt.month
precio_daily["year"] = precio_daily["date"].dt.year

monthly = precio_daily.groupby(["year", "month"])["precio_daily_avg"].mean().reset_index()
monthly["date"] = pd.to_datetime(monthly[["year", "month"]].assign(day=1))

alt.Chart(monthly).mark_bar().encode(
    x=alt.X("month:O", title="Month"),
    y=alt.Y("mean(precio_daily_avg):Q", title="Avg price (COP/kWh)"),
).properties(width=500, height=300, title="Average spot price by month (2000-2026)")

alt.Chart(...)

## 5. Hydrology

### 5.1 Reservoir levels

In [60]:
reservoir_level = alt.Chart(volumen).mark_line(strokeWidth=0.5).encode(
    x=alt.X("date:T", title=""),
    y=alt.Y("volumen_util_pct:Q", title="Useful volume (%)"),
).properties(width=800, height=250, title="Aggregate reservoir level (2000-2026)")

reservoir_level

alt.Chart(...)

In [61]:
reservoir_level.save("../figures/eda/reservoir_level.png", scale_factor=2.0)

### 5.2 Hydro inflows

In [62]:
aportes["aportes_gwh"] = aportes["aportes_kwh"] / 1e6

daily_hydro_inflows = alt.Chart(aportes).mark_line(strokeWidth=0.3).encode(
    x=alt.X("date:T", title=""),
    y=alt.Y("aportes_gwh:Q", title="Inflows (GWh)"),
).properties(width=800, height=250, title="Daily hydro inflows (2000-2026)")

daily_hydro_inflows

alt.Chart(...)

In [63]:
daily_hydro_inflows.save("../figures/eda/daily_hydro_inflows.png", scale_factor=2.0)

## 6. Spot price vs hydrology

Overlay spot price with reservoir levels to visually confirm that price regimes
align with hydrological stress.

In [64]:
merged = precio_daily[["date", "precio_daily_avg"]].merge(
    volumen[["date", "volumen_util_pct"]], on="date", how="inner"
)

base = alt.Chart(merged).encode(x=alt.X("date:T", title=""))

price_line = base.mark_line(strokeWidth=0.5, color="#e45756").encode(
    y=alt.Y("precio_daily_avg:Q", title="Spot price (COP/kWh)"),
)

vol_line = base.mark_line(strokeWidth=0.5, color="#4c78a8").encode(
    y=alt.Y("volumen_util_pct:Q", title="Reservoir level (%)"),
)

price_vs_reservoir = alt.layer(price_line, vol_line).resolve_scale(y="independent").properties(
    width=800, height=350, title="Spot price (red) vs reservoir level (blue)"
)

price_vs_reservoir

alt.LayerChart(...)

In [65]:
price_vs_reservoir.save("../figures/eda/price_vs_reservoir.png", scale_factor=2.0)

In [66]:
price_vs_reservoir_scatter = alt.Chart(merged).mark_circle(size=3, opacity=0.3).encode(
    x=alt.X("volumen_util_pct:Q", title="Reservoir level (%)"),
    y=alt.Y("precio_daily_avg:Q", title="Spot price (COP/kWh)"),
).properties(width=500, height=400, title="Price vs reservoir level")

price_vs_reservoir_scatter

alt.Chart(...)

In [67]:
price_vs_reservoir_scatter.save("../figures/eda/price_vs_reservoir_scatter.png", scale_factor=2.0)

In [68]:
corr = merged[["precio_daily_avg", "volumen_util_pct"]].corr().iloc[0, 1]
print(f"Pearson correlation (price vs reservoir): {corr:.3f}")

Pearson correlation (price vs reservoir): -0.240


## 7. Generation and demand

In [69]:
gen_daily = generacion.set_index("datetime").resample("D")["generacion_kwh"].sum().reset_index()
gen_daily.columns = ["date", "gen_daily_kwh"]
gen_daily["gen_daily_gwh"] = gen_daily["gen_daily_kwh"] / 1e6

dem_daily = demanda.set_index("datetime").resample("D")["demanda_kwh"].sum().reset_index()
dem_daily.columns = ["date", "dem_daily_kwh"]
dem_daily["dem_daily_gwh"] = dem_daily["dem_daily_kwh"] / 1e6

gen_dem = gen_daily[["date", "gen_daily_gwh"]].merge(
    dem_daily[["date", "dem_daily_gwh"]], on="date"
)

gen_line = alt.Chart(gen_dem).mark_line(strokeWidth=0.4, color="#4c78a8").encode(
    x="date:T", y=alt.Y("gen_daily_gwh:Q", title="GWh/day"),
)
dem_line = alt.Chart(gen_dem).mark_line(strokeWidth=0.4, color="#e45756").encode(
    x="date:T", y=alt.Y("dem_daily_gwh:Q"),
)

generation_vs_demand = (gen_line + dem_line).properties(
    width=800, height=300, title="Daily generation (blue) vs demand (red)"
)

generation_vs_demand

alt.LayerChart(...)

In [70]:
generation_vs_demand.save("../figures/eda/generation_vs_demand.png", scale_factor=2.0)

## 8. Year-over-year price comparison

In [71]:
precio_daily["day_of_year"] = precio_daily["date"].dt.dayofyear

years_of_interest = [2002, 2009, 2015, 2023, 2024, 2025]
yoy = precio_daily[precio_daily["year"].isin(years_of_interest)]

yoy_el_nino = alt.Chart(yoy).mark_line(strokeWidth=1).encode(
    x=alt.X("day_of_year:Q", title="Day of year", scale=alt.Scale(domain=[1, 366])),
    y=alt.Y("precio_daily_avg:Q", title="Spot price (COP/kWh)"),
    color=alt.Color("year:N", title="Year"),
).properties(width=700, height=350, title="Spot price by day of year (El Niño years)")

yoy_el_nino

alt.Chart(...)

In [72]:
yoy_el_nino.save("../figures/eda/yoy_el_nino.png", scale_factor=2.0)

## 9. Summary statistics for price

In [73]:
from prettytable import PrettyTable

stats = precio_daily["precio_daily_avg"].describe()
t = PrettyTable(["Statistic", "Value"])
for k, v in stats.items():
    t.add_row([k, f"{v:,.2f}"])
t.add_row(["skewness", f"{precio_daily['precio_daily_avg'].skew():,.2f}"])
t.add_row(["kurtosis", f"{precio_daily['precio_daily_avg'].kurt():,.2f}"])
print(t)

+-----------+----------+
| Statistic |  Value   |
+-----------+----------+
|   count   | 9,709.00 |
|    mean   |  184.43  |
|    std    |  213.05  |
|    min    |  28.84   |
|    25%    |  70.29   |
|    50%    |  113.00  |
|    75%    |  196.04  |
|    max    | 2,498.80 |
|  skewness |   3.69   |
|  kurtosis |  20.46   |
+-----------+----------+


## Key observations

1. Price distribution:
2. Seasonality patterns:
3. Hydrology correlation:
4. Visible regime candidates:
5. Data quality issues: